# Assignment 2 — Deep Learning on Cityscapes (Google Colab Runner)

**Course:** CO3133 — Deep Learning and Its Applications (Semester 261)  
**Instructor:** Dr. Le Thanh Sach  
**Group:** RTX5090  
**Repository:** [CO3133_sem261_groupRTX5090](https://github.com/phsuuuuuu/CO3133_sem261_groupRTX5090)  

---
### Overview
This notebook provides an automated, end-to-end setup to execute the **three DeepLabV3 semantic segmentation experiments** described in `command.md` on Google Colab.

#### Key Highlights:
1. **Top-Level Hyperparameter Controls**: Customize epochs, batch sizing, learning rates, random seed, and quick-debug limits right at the top of the notebook.
2. **Zero-Config GPU Auto-Tuning**: Automatically detects your allocated GPU (**Tesla T4**, **NVIDIA L4**, or **NVIDIA A100**) and tunes batch size, dataloader workers, and AMP for maximum performance without manual intervention.
3. **Multi-User Ready**: Configure your own Git credentials in the form fields so commit history reflects your identity.
4. **Local Config Sandbox**: Config files modified on Colab stay strictly local and are **never pushed to GitHub**.
5. **Execute 3 Experiments**:
   * **Experiment 1:** Frozen backbone (train segmentation head only)
   * **Experiment 2:** Partial fine-tuning (backbone `layer3` + `layer4` + head)
   * **Experiment 3:** Full fine-tuning (backbone + head)
6. **Auto-Push to GitHub**: Automatically commits and pushes only `log.csv` and `metrics_test.json` to GitHub (heavy model weights `.pt` and local configs are excluded).
7. **Drive Backup & Local Download**: Save checkpoints to Google Drive and download CSV logs directly.

## 🎛️ Top-Level Training Hyperparameters & Experiment Controls
Adjust the parameters below to match your experimental requirements.

* **`BATCH_SIZE_MODE`**: Set to `Auto (Detect GPU)` to let the notebook automatically pick the optimal batch size (16 for A100, 12 for L4, 8 for T4), or choose `Custom` to set your own.
* **`MAX_TRAIN_BATCHES`**: Set to `0` for full training on all 2,481 images. Set to e.g. `50` for a fast 1-minute sanity test/dry run.

In [ ]:
# @title 🎛️ Hyperparameter Control Panel
# @markdown ### General Experiment Settings:
EPOCHS = 10  # @param {type:"integer"}
RANDOM_SEED = 666  # @param {type:"integer"}
EVAL_EVERY = 1  # @param {type:"integer"}

# @markdown ### Hardware & Batch Sizing:
BATCH_SIZE_MODE = "Auto (Detect GPU)"  # @param ["Auto (Detect GPU)", "Custom"]
CUSTOM_BATCH_SIZE = 8  # @param {type:"integer"}
CUSTOM_NUM_WORKERS = 4  # @param {type:"integer"}
USE_AMP = True  # @param {type:"boolean"}

# @markdown ### Learning Rates:
HEAD_LR = 0.001  # @param {type:"number"}
BACKBONE_LR_PARTIAL = 0.0001  # @param {type:"number"}
BACKBONE_LR_FULL = 0.00001  # @param {type:"number"}

# @markdown ### Fast Testing / Debugging Mode:
# @markdown Set batches > 0 for a quick dry run (e.g. 50 batches instead of full dataset). Set 0 for complete training.
MAX_TRAIN_BATCHES = 0  # @param {type:"integer"}
MAX_EVAL_BATCHES = 0  # @param {type:"integer"}

print("=" * 65)
print("🎛️  TRAINING CONTROLS INITIALIZED:")
print(f"   • Target Epochs        : {EPOCHS}")
print(f"   • Batch Size Mode      : {BATCH_SIZE_MODE} (Custom={CUSTOM_BATCH_SIZE})")
print(f"   • Head Learning Rate   : {HEAD_LR}")
print(f"   • Partial Backbone LR  : {BACKBONE_LR_PARTIAL}")
print(f"   • Full Backbone LR     : {BACKBONE_LR_FULL}")
print(f"   • Random Seed          : {RANDOM_SEED}")
print(f"   • Mixed Precision AMP  : {USE_AMP}")
print(f"   • Train Batches Limit  : {'Full Dataset' if MAX_TRAIN_BATCHES == 0 else f'{MAX_TRAIN_BATCHES} batches (Dry Run)'}")
print("=" * 65)

## Step 1: Verify Hardware Accelerator
Ensure your Colab runtime is set to GPU (**Runtime** > **Change runtime type** > **GPU** -> Select **T4**, **L4**, or **A100**).

In [ ]:
# @title Check GPU Specifications
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## Step 2: Clone Repository & Install Dependencies
This cell clones the repository into `/content/CO3133_sem261_groupRTX5090`, navigates into the `assignment2` directory, and installs the required libraries (`cityscapesscripts`, `torchmetrics`, `PyYAML`, `tqdm`).

In [ ]:
# @title Clone Repository and Setup Workspace
import os

REPO_URL = "https://github.com/phsuuuuuu/CO3133_sem261_groupRTX5090.git"
ROOT_DIR = "/content/CO3133_sem261_groupRTX5090"
ASSIGNMENT_DIR = f"{ROOT_DIR}/assignment2"

if not os.path.exists(ROOT_DIR):
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL} {ROOT_DIR}
else:
    print("Repository already exists. Pulling latest commits...")
    !git -C {ROOT_DIR} checkout main
    !git -C {ROOT_DIR} pull origin main

%cd {ASSIGNMENT_DIR}

# Install required packages (preserving Colab's native PyTorch + CUDA)
!pip install -q cityscapesscripts torchmetrics PyYAML tqdm

## Step 3: Configure Your GitHub Credentials (for Auto-Pushing Logs)
Each team member using this notebook should enter their own **Name**, **Email**, and **Personal Access Token (PAT)** below so git commits are accurately attributed.

> **Safety Notice:** This auto-push mechanism **ONLY** stages and pushes `log.csv` and `metrics_*.json`. It will **NEVER** push local YAML configs or heavy model weights (`.pt`).

**How to get your GitHub Token (takes 30 seconds):**
1. Go to [GitHub Token Settings (classic)](https://github.com/settings/tokens).
2. Click **Generate new token (classic)**.
3. Set a name (e.g., `colab-runner`), choose an expiration, and check the **`repo`** scope checkbox.
4. Copy your token and paste it into `GITHUB_TOKEN` below (or store it in Colab's **Secrets 🔑** tab as `GITHUB_TOKEN`).

In [ ]:
# @title GitHub User Identity & Auto-Push Setup
import os
import subprocess

# Attempt to load token from Colab Secrets (key icon on left sidebar)
token_from_secrets = None
try:
    from google.colab import userdata
    token_from_secrets = userdata.get('GITHUB_TOKEN')
except Exception:
    pass

# @markdown **Enter your GitHub Credentials:**
GITHUB_TOKEN = token_from_secrets or ""  # @param {type:"string"}
GIT_USERNAME = ""  # @param {type:"string"}
GIT_EMAIL = ""  # @param {type:"string"}
REPO_OWNER = "phsuuuuuu"  # @param {type:"string"}
REPO_NAME = "CO3133_sem261_groupRTX5090"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}

def auto_push_logs(commit_message="Update training logs and metrics from Colab"):
    """
    Automatically stages log.csv and metrics_*.json and pushes them to GitHub.
    Config files (configs/*.yaml) and model weights (*.pt) are NEVER pushed.
    """
    if not GITHUB_TOKEN:
        print("[NOTICE] GITHUB_TOKEN is not provided. Skipping auto-push to GitHub.")
        print("You can still download logs locally in the final step.")
        return False

    author_name = GIT_USERNAME.strip() or "Colab Contributor"
    author_email = GIT_EMAIL.strip() or "colab@google.com"

    %cd {ASSIGNMENT_DIR}
    !git config user.name "{author_name}"
    !git config user.email "{author_email}"

    # Stage ONLY log.csv and metrics_test.json files
    !git add checkpoints/**/log.csv checkpoints/**/metrics_*.json

    # Check if there are changes to commit
    status = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True).stdout
    staged_lines = [line for line in status.splitlines() if line.startswith(('M ', 'A ', 'AM'))]
    
    if not staged_lines:
        print("[GIT] No changes in logs or metrics to commit.")
        return True

    print("[GIT] Staged files for commit:")
    for line in staged_lines:
        print(" ", line)
    
    !git commit -m "{commit_message}"
    
    remote_url = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
    print("[GIT] Pulling latest remote changes with rebase...")
    !git pull --rebase "{remote_url}" {BRANCH}
    
    print(f"[GIT] Pushing logs as '{author_name}' to {REPO_OWNER}/{REPO_NAME} ({BRANCH})...")
    res = subprocess.run(["git", "push", remote_url, BRANCH], capture_output=True, text=True)
    if res.returncode == 0:
        print("[SUCCESS] Training logs successfully pushed to GitHub!")
        return True
    else:
        print("[ERROR] Failed to push to GitHub:")
        print(res.stderr)
        return False

if GITHUB_TOKEN:
    print(f"\u2705 GitHub credentials configured for user: {GIT_USERNAME or 'Anonymous'}. Auto-push is ENABLED.")
else:
    print("\u26a0\ufe0f GITHUB_TOKEN is empty. Paste your token above to enable automatic GitHub pushes.")

## Step 4: Dataset Preparation
Choose **Option A** (Recommended if you have uploaded archives/folders to Google Drive) or **Option B** (Direct download from Cityscapes portal).

In [ ]:
# @title Option A (Recommended): Mount Google Drive and Connect Dataset
# Upload your Cityscapes archives or extracted folders to MyDrive/Cityscapes/
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/Cityscapes"
LOCAL_DATASET = f"{ASSIGNMENT_DIR}/dataset"
os.makedirs(LOCAL_DATASET, exist_ok=True)

# Check for unzipped directories on Drive
if os.path.isdir(f"{DRIVE_DIR}/leftImg8bit_trainvaltest") and os.path.isdir(f"{DRIVE_DIR}/gtFine_trainvaltest"):
    print("Found unzipped dataset on Google Drive. Linking via symlinks...")
    !ln -sfn "{DRIVE_DIR}/leftImg8bit_trainvaltest" "{LOCAL_DATASET}/leftImg8bit_trainvaltest"
    !ln -sfn "{DRIVE_DIR}/gtFine_trainvaltest" "{LOCAL_DATASET}/gtFine_trainvaltest"
    print("Symlinks created successfully!")

# Check for zip archives on Drive
elif os.path.isfile(f"{DRIVE_DIR}/leftImg8bit_trainvaltest.zip") and os.path.isfile(f"{DRIVE_DIR}/gtFine_trainvaltest.zip"):
    print("Found zip archives on Google Drive. Extracting to local fast SSD (approx 1-2 min)...")
    if not os.path.isdir(f"{LOCAL_DATASET}/gtFine_trainvaltest"):
        print("Extracting gtFine_trainvaltest.zip...")
        !unzip -q "{DRIVE_DIR}/gtFine_trainvaltest.zip" -d "{LOCAL_DATASET}/"
    if not os.path.isdir(f"{LOCAL_DATASET}/leftImg8bit_trainvaltest"):
        print("Extracting leftImg8bit_trainvaltest.zip...")
        !unzip -q "{DRIVE_DIR}/leftImg8bit_trainvaltest.zip" -d "{LOCAL_DATASET}/"
    print("Dataset extraction complete!")
else:
    print(f"[WARNING] Dataset not found in {DRIVE_DIR}.")
    print("Please verify your Drive folder structure or use Option B below.")

In [ ]:
# @title Option B (Alternative): Download Directly from Cityscapes Portal
# Only run this if you do NOT have the dataset on Google Drive.
# Enter your Cityscapes portal credentials (https://www.cityscapes-dataset.com/)
CITYSCAPES_USER = ""  # @param {type:"string"}
CITYSCAPES_PASS = ""  # @param {type:"string"}

if CITYSCAPES_USER and CITYSCAPES_PASS:
    %cd {ASSIGNMENT_DIR}/dataset
    print("Downloading gtFine_trainvaltest.zip via high-speed Colab network...")
    !csDownload gtFine_trainvaltest.zip -u "{CITYSCAPES_USER}" -p "{CITYSCAPES_PASS}"
    !unzip -q gtFine_trainvaltest.zip

    print("Downloading leftImg8bit_trainvaltest.zip (11 GB)...")
    !csDownload leftImg8bit_trainvaltest.zip -u "{CITYSCAPES_USER}" -p "{CITYSCAPES_PASS}"
    !unzip -q leftImg8bit_trainvaltest.zip
    
    %cd {ASSIGNMENT_DIR}
    print("Direct download and extraction finished!")
else:
    print("Skipping Option B (credentials not entered).")

In [ ]:
# @title Verify trainIds Masks & Pre-split Partitions
# Converts 34 raw Cityscapes labelIds into 19 training classes (with ignore_index=255)
import glob

%cd {ASSIGNMENT_DIR}
train_masks = glob.glob(f"{ASSIGNMENT_DIR}/dataset/**/*_gtFine_trainIds.png", recursive=True)

if len(train_masks) == 0:
    print("Generating trainIds masks using scripts/prepare_data.py...")
    !python scripts/prepare_data.py
else:
    print(f"Verified: Found {len(train_masks)} trainIds mask files. Ready to train!")

## Step 5: Automatic Hardware Detection & Applying Hyperparameters

This cell **automatically inspects your allocated hardware**, binds your hyperparameters from the top panel, and updates the local YAML files on this Colab VM.

### Benchmark Profile Comparison:

| Detected GPU | VRAM & Specs | Auto Batch Size | Auto Workers | Mixed Precision | Est. Time / Model (10 Ep) | **Est. Time (All 3 Models)** |
| :--- | :--- | :---: | :---: | :---: | :---: | :---: |
| **Tesla T4** | 16 GB GDDR6 (2 vCPUs) | **`8`** | **`2`** | FP16 AMP | ~35 – 40 min | **~1.8 – 2.0 hours** |
| **NVIDIA L4** | 24 GB GDDR6 (4–8 vCPUs) | **`12`** | **`4`** | FP16/BF16 AMP | ~12 – 14 min | **~38 – 45 min** |
| **NVIDIA A100** | 40/80 GB HBM2e (8–12 vCPUs) | **`16`** | **`4`** | FP16/BF16 AMP | ~7 – 8 min | **~22 – 25 min** |
| *(Local RTX 5070 Ti)* | 16 GB GDDR7 (Blackwell) | `8` | `4` | FP16 AMP | ~12 min | ~36 min |

> **Safety Guarantee:** The parameters are updated directly in `configs/*.yaml` **inside this Colab VM only**. They will **NEVER be pushed to GitHub**.

In [ ]:
# @title Apply Optimal Settings and User Hyperparameters
import os
import torch
import yaml
from pathlib import Path

cpu_cores = os.cpu_count() or 2

if not torch.cuda.is_available():
    detected_gpu_title = "CPU Fallback (No CUDA)"
    auto_batch_size = 4
    auto_workers = 2
    est_per_epoch = "~15 - 20 minutes"
    est_total = "~8 - 10 hours"
else:
    raw_device_name = torch.cuda.get_device_name(0).upper()
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    
    if "A100" in raw_device_name:
        detected_gpu_title = f"NVIDIA A100 Tensor Core ({vram_gb:.1f} GB VRAM, HBM2e)"
        auto_batch_size = 16
        auto_workers = min(cpu_cores, 4)
        est_per_epoch = "~40 - 50 seconds"
        est_total = "~22 - 25 minutes (all 3 models)"
    elif "L4" in raw_device_name:
        detected_gpu_title = f"NVIDIA L4 Tensor Core ({vram_gb:.1f} GB VRAM, Ada Lovelace)"
        auto_batch_size = 12
        auto_workers = min(cpu_cores, 4)
        est_per_epoch = "~1.2 - 1.4 minutes"
        est_total = "~38 - 45 minutes (all 3 models)"
    elif "T4" in raw_device_name:
        detected_gpu_title = f"NVIDIA Tesla T4 ({vram_gb:.1f} GB VRAM, Turing)"
        auto_batch_size = 8
        auto_workers = min(cpu_cores, 2)
        est_per_epoch = "~3.5 - 4.0 minutes"
        est_total = "~1.8 - 2.0 hours (all 3 models)"
    elif "V100" in raw_device_name:
        detected_gpu_title = f"NVIDIA Tesla V100 ({vram_gb:.1f} GB VRAM, Volta)"
        auto_batch_size = 12
        auto_workers = min(cpu_cores, 4)
        est_per_epoch = "~1.0 - 1.2 minutes"
        est_total = "~30 - 35 minutes (all 3 models)"
    else:
        detected_gpu_title = f"{raw_device_name} ({vram_gb:.1f} GB VRAM)"
        auto_batch_size = 16 if vram_gb >= 30 else (12 if vram_gb >= 20 else (8 if vram_gb >= 12 else 4))
        auto_workers = min(cpu_cores, 4)
        est_per_epoch = "Adaptive"
        est_total = "Adaptive"

# Resolve batch size and workers based on top panel settings
if BATCH_SIZE_MODE == "Auto (Detect GPU)":
    final_batch_size = auto_batch_size
    final_workers = auto_workers
else:
    final_batch_size = CUSTOM_BATCH_SIZE
    final_workers = CUSTOM_NUM_WORKERS

print("=" * 75)
print(f"\U0001f916 AUTO-DETECTED HARDWARE: {detected_gpu_title}")
print(f"   Available CPU Cores     : {cpu_cores}")
print("-" * 75)
print(f"\u2699\ufe0f  ACTIVE HYPERPARAMETERS (Injected from Top Panel):")
print(f"   • Batch Size             : {final_batch_size} ({'Auto-Tuned' if BATCH_SIZE_MODE == 'Auto (Detect GPU)' else 'Custom'})")
print(f"   • Dataloader Workers     : {final_workers}")
print(f"   • Total Epochs           : {EPOCHS}")
print(f"   • Head Learning Rate     : {HEAD_LR}")
print(f"   • Partial Backbone LR    : {BACKBONE_LR_PARTIAL}")
print(f"   • Full Backbone LR       : {BACKBONE_LR_FULL}")
print(f"   • Random Seed            : {RANDOM_SEED}")
print(f"   • Mixed Precision (AMP)  : {USE_AMP}")
print(f"   • Max Batches Limit      : {'Full Dataset' if MAX_TRAIN_BATCHES == 0 else f'{MAX_TRAIN_BATCHES} batches (Dry Run)'}")
print("-" * 75)
print(f"\u23f1\ufe0f  BENCHMARK TIME ESTIMATE:")
print(f"   • Estimated / Epoch      : {est_per_epoch}")
print(f"   • Total (All 3 Models)   : {est_total}")
print("=" * 75)

# Update local YAML files in-place on this Colab VM
target_configs = [
    ("configs/deeplabv3.yaml", HEAD_LR, None),
    ("configs/resnet_layer3_4_finetune.yaml", HEAD_LR, BACKBONE_LR_PARTIAL),
    ("configs/resnet50_full_deeplabv3.yaml", HEAD_LR, BACKBONE_LR_FULL)
]

for cfg_rel, head_lr_val, backbone_lr_val in target_configs:
    cfg_file = Path(ASSIGNMENT_DIR) / cfg_rel
    if cfg_file.exists():
        with open(cfg_file, "r", encoding="utf-8") as f:
            config_data = yaml.safe_load(f)
        
        config_data["batch_size"] = final_batch_size
        config_data["num_workers"] = final_workers
        config_data["epochs"] = EPOCHS
        config_data["seed"] = RANDOM_SEED
        config_data["amp"] = USE_AMP
        config_data["pin_memory"] = True
        config_data["eval_every"] = EVAL_EVERY
        config_data["max_train_batches"] = MAX_TRAIN_BATCHES
        config_data["max_eval_batches"] = MAX_EVAL_BATCHES
        
        config_data["lr"] = head_lr_val
        config_data["head_lr"] = head_lr_val
        if backbone_lr_val is not None:
            config_data["backbone_lr"] = backbone_lr_val
        
        with open(cfg_file, "w", encoding="utf-8") as f:
            yaml.dump(config_data, f, default_flow_style=False, sort_keys=False)
        
        print(f"  \u2713 Configured {cfg_rel}: batch_size={final_batch_size}, epochs={EPOCHS}, lr={head_lr_val}")

print("\n[STATUS] Configuration updated for current Colab session only. GitHub repository remains completely untouched.")

## Step 6: Run the Three DeepLabV3 Experiments

We train each model sequentially for the configured epochs as defined in `command.md`:
| Experiment | Description | Config File | Output Checkpoint Dir |
| :--- | :--- | :--- | :--- |
| **Model 1** | Frozen ImageNet Backbone (Train Head Only) | `configs/deeplabv3.yaml` | `checkpoints/deeplabv3/local/` |
| **Model 2** | Partial Fine-Tuning (`layer3` + `layer4` + Head) | `configs/resnet_layer3_4_finetune.yaml` | `checkpoints/deeplabv3/resnet_layer3_4_finetune/` |
| **Model 3** | Full Fine-Tuning (All Layers + Head) | `configs/resnet50_full_deeplabv3.yaml` | `checkpoints/deeplabv3/resnet50_full_deeplabv3/` |

> **Auto-Push Feature:** After each experiment completes, its `log.csv` is committed and pushed directly to GitHub so you never lose progress even if Colab disconnects.

In [ ]:
# @title Model 1: Frozen Backbone Training & Auto-Push
# Trains only the ASPP classifier head while freezing the ResNet-50 backbone.
%cd {ASSIGNMENT_DIR}
!python -m src.train --config configs/deeplabv3.yaml

# Auto-push log.csv to GitHub immediately
auto_push_logs("Colab: update Model 1 (Frozen Backbone) training logs")

In [ ]:
# @title Model 2: Partial Fine-Tuning (Layers 3 & 4) & Auto-Push
# Freezes layers 1 & 2; fine-tunes layers 3 & 4 and the classifier head.
%cd {ASSIGNMENT_DIR}
!python -m src.train --config configs/resnet_layer3_4_finetune.yaml

# Auto-push log.csv to GitHub immediately
auto_push_logs("Colab: update Model 2 (Partial Fine-tuning) training logs")

In [ ]:
# @title Model 3: Full Fine-Tuning & Auto-Push
# Fine-tunes all backbone layers and the classifier head with differential learning rates.
%cd {ASSIGNMENT_DIR}
!python -m src.train --config configs/resnet50_full_deeplabv3.yaml

# Auto-push log.csv to GitHub immediately
auto_push_logs("Colab: update Model 3 (Full Fine-tuning) training logs")

## Step 7: Evaluate Best Checkpoints on the Test Set & Auto-Push Metrics
Evaluate each experiment's `best.pt` on the official test partition, export metrics to `metrics_test.json`, and push the results to GitHub.

In [ ]:
# @title Evaluate All 3 Models on the Test Split
%cd {ASSIGNMENT_DIR}

print("=== [1/3] Evaluating Model 1 (Frozen Backbone) ===")
!python -m src.evaluate --config configs/deeplabv3.yaml \
    --checkpoint checkpoints/deeplabv3/local/best.pt \
    --split test --max_batches 0 \
    --save_json checkpoints/deeplabv3/local/metrics_test.json

print("\n=== [2/3] Evaluating Model 2 (Partial Fine-Tuning) ===")
!python -m src.evaluate --config configs/resnet_layer3_4_finetune.yaml \
    --checkpoint checkpoints/deeplabv3/resnet_layer3_4_finetune/best.pt \
    --split test --max_batches 0 \
    --save_json checkpoints/deeplabv3/resnet_layer3_4_finetune/metrics_test.json

print("\n=== [3/3] Evaluating Model 3 (Full Fine-Tuning) ===")
!python -m src.evaluate --config configs/resnet50_full_deeplabv3.yaml \
    --checkpoint checkpoints/deeplabv3/resnet50_full_deeplabv3/best.pt \
    --split test --max_batches 0 \
    --save_json checkpoints/deeplabv3/resnet50_full_deeplabv3/metrics_test.json

# Push all test evaluation JSON metrics to GitHub
auto_push_logs("Colab: upload final test evaluation metrics_test.json for all 3 models")

In [ ]:
# @title Summary Comparison Table of Results
import json
import pandas as pd

models = [
    ("Model 1: Frozen Backbone", "checkpoints/deeplabv3/local/metrics_test.json"),
    ("Model 2: Partial Fine-Tuning", "checkpoints/deeplabv3/resnet_layer3_4_finetune/metrics_test.json"),
    ("Model 3: Full Fine-Tuning", "checkpoints/deeplabv3/resnet50_full_deeplabv3/metrics_test.json"),
]

summary = []
for label, path in models:
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        summary.append({
            "Experiment": label,
            "Test mIoU (%)": f"{data.get('miou', 0) * 100:.2f}%",
            "Mean Dice (%)": f"{data.get('mdice', 0) * 100:.2f}%",
            "Pixel Acc (%)": f"{data.get('pixel_acc', 0) * 100:.2f}%",
            "Test Loss": f"{data.get('loss', 0):.4f}",
            "Eval Time (s)": f"{data.get('eval_time_sec', 0):.1f}",
        })

if summary:
    df = pd.DataFrame(summary)
    display(df)
else:
    print("Metrics test JSON files not found yet. Please run evaluation first.")

## Step 8: Backup Model Weights to Drive & Local Download
Save the generated `.pt` model weights directly to your Google Drive to keep checkpoints safe, and download CSV logs directly to your local PC.

In [ ]:
# @title Save Heavy Checkpoints (*.pt) to Google Drive
import shutil

BACKUP_DRIVE_DIR = "/content/drive/MyDrive/Cityscapes_Checkpoints"
os.makedirs(BACKUP_DRIVE_DIR, exist_ok=True)

print(f"Copying checkpoints to {BACKUP_DRIVE_DIR}...")
!cp -r checkpoints/* "{BACKUP_DRIVE_DIR}/"
print("All checkpoint files and weights backed up successfully to Google Drive!")

In [ ]:
# @title Download CSV Logs Directly to Your Computer (Browser Download)
from google.colab import files

csv_files = [
    "checkpoints/deeplabv3/local/log.csv",
    "checkpoints/deeplabv3/resnet_layer3_4_finetune/log.csv",
    "checkpoints/deeplabv3/resnet50_full_deeplabv3/log.csv",
]

for f_path in csv_files:
    if os.path.exists(f_path):
        print(f"Downloading {f_path}...")
        files.download(f_path)
    else:
        print(f"File not found: {f_path}")